# 03 – Clinical Note Availability

Analyses how many clinical notes are available for the cohort admissions (whole-admission
counts, by phenotype), motivating the text-modality analysis.

**Run after notebook 02** — uses final_cohort_angus.csv and the raw NOTEEVENTS table.

**Produces:** no files (analysis and summary statistics only).

MIMIC-III data not included (PhysioNet DUA); see README. Patient-row outputs cleared.

In [ ]:
# --- Setup ---
import os

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

import pandas as pd
from collections import Counter


In [ ]:
import pandas as pd

cohort = pd.read_csv(
    data_path("final_cohort_angus.csv")
)

print(cohort.shape)
cohort.head()

In [ ]:
cohort.columns.tolist()

['ROW_ID',
 'SUBJECT_ID',
 'HADM_ID',
 'ICUSTAY_ID',
 'DBSOURCE',
 'FIRST_CAREUNIT',
 'LAST_CAREUNIT',
 'FIRST_WARDID',
 'LAST_WARDID',
 'INTIME',
 'OUTTIME',
 'LOS',
 'Sepsis_Angus',
 'AKI',
 'GENDER',
 'DOB',
 'ADMITTIME',
 'DISCHTIME',
 'DEATHTIME',
 'ADMISSION_TYPE',
 'ETHNICITY',
 'HOSPITAL_EXPIRE_FLAG',
 'HAS_CHARTEVENTS_DATA',
 'AGE']

In [ ]:
hadm_set = set(cohort["HADM_ID"])

print(len(hadm_set))

33560


In [ ]:
category_counter = Counter()

notes_path = data_path("NOTEEVENTS.csv.gz")

for chunk in pd.read_csv(
    notes_path,
    usecols=["HADM_ID", "CATEGORY"],
    chunksize=100000
):
    chunk = chunk.dropna(subset=["HADM_ID"])

    matched = chunk[
        chunk["HADM_ID"].isin(hadm_set)
    ]

    category_counter.update(
        matched["CATEGORY"].dropna()
    )

category_counts = (
    pd.Series(category_counter)
    .sort_values(ascending=False)
)

category_counts

,0
Nursing/other,327750
Radiology,273251
Nursing,155994
ECG,96067
Physician,95376
Discharge summary,37441
Echo,24663
Respiratory,23796
Nutrition,6917
General,5424


In [ ]:
notes_counter = Counter()

notes_path = data_path("NOTEEVENTS.csv.gz")

for chunk in pd.read_csv(
    notes_path,
    usecols=["HADM_ID"],
    chunksize=100000
):
    chunk = chunk.dropna(subset=["HADM_ID"])

    matched = chunk[
        chunk["HADM_ID"].isin(hadm_set)
    ]

    notes_counter.update(
        matched["HADM_ID"].astype(int)
    )

In [ ]:
cohort["num_notes"] = (
    cohort["HADM_ID"]
    .map(notes_counter)
    .fillna(0)
    .astype(int)
)

In [ ]:
cohort.groupby("Sepsis_Angus")["num_notes"].describe()

,count,mean,std,min,25%,50%,75%,max
Sepsis_Angus,,,,,,,,
0,23436.0,20.938556,23.291388,0.0,9.0,14.0,24.0,445.0
1,10124.0,55.601146,74.846346,0.0,15.0,31.0,66.0,1233.0


In [ ]:
cohort.groupby("AKI")["num_notes"].describe()

,count,mean,std,min,25%,50%,75%,max
AKI,,,,,,,,
0,26213.0,26.463549,35.434039,0.0,10.0,16.0,29.0,704.0
1,7347.0,48.990472,75.697441,0.0,13.0,25.0,51.0,1233.0


In [ ]:
cohort.groupby("Sepsis_Angus")["num_notes"].apply(
    lambda x: (x > 0).mean()
)

,num_notes
Sepsis_Angus,
0,0.995562
1,0.990320


In [ ]:
cohort.groupby("AKI")["num_notes"].apply(
    lambda x: (x > 0).mean()
)

,num_notes
AKI,
0,0.995117
1,0.989928
